# 🦅 Áquila Pro — A100 Sovereign Neural Engine
Este notebook executa o motor neural de alta performance em **GPU NVIDIA A100** para processamento de tarefas complexas do Áquila Pro.

### ⚡ Configuração Inicial de Runtime:
1. No menu superior: **Ambiente de execução (Runtime)** > **Alterar tipo de ambiente de execução**.
2. Selecione **GPU** e escolha **A100 GPU** com perfil **High-RAM**.

## Caminho unico de execucao (revisado)

1. Em uma sessao limpa do Colab, faca upload de `colab/aquila_hardening.py` pelo painel Arquivos; o Colab o coloca em `/content`. As celulas 10 e 27 validam `Path.cwd()/aquila_hardening.py` antes de importar o modulo, sem depender de pastas temporarias da sessao. A celula 1 verifica hardware; a celula 4 instala dependencias usadas pelo treinamento.
2. Antes de acessar o dataset remoto, execute localmente em `colab/`: `python -m unittest -v test_aquila_hardening`. O teste do manifesto usa somente `fixtures/aquila_manifest_fixture.csv`; depois execute a celula 10 para validar o dataset remoto e gravar `aquila_training_manifest.json`. Ela e somente auditoria por padrao: nao carrega o modelo nem treina. O relatorio traz contagens, motivos de exclusao e IDs dos registros elegiveis ao filtro antigo que gerava resolucoes genericas; isso nao afirma quais registros foram efetivamente treinados anteriormente. O manifesto registra a disposicao de cada registro.
3. Somente se a tarefa desejada for **previsao do gabarito** (nao resolucao didatica), disponibilize `AQUILA_ENABLE_TRAINING=1` no ambiente/runtime e execute a celula 10 uma unica vez. Ela bloqueia reexecucao de treinamento na mesma sessao. Nao execute as antigas celulas 5-9 ou 14; elas sao fluxos historicos desativados.
4. Apos disponibilizar o diretorio `aquila_tutor_model`, para servir a API configure `AQUILA_INFERENCE_TOKEN` como segredo de runtime (minimo 32 caracteres), execute a celula 26 se necessario e depois a celula 27. CORS aceita somente origens explicitamente listadas em `AQUILA_ALLOWED_ORIGINS`; isso nao substitui autenticacao. Benchmark fica desabilitado em modo publico.
5. A celula 28 e apenas uma etapa opcional de exposicao: requer API configurada em modo publico e `AQUILA_ALLOW_PUBLIC_TUNNEL=1`; ela nao instala nem encerra processos. Nao a execute sem autorizacao explicita para publicar o servico.

As saidas ja existentes foram preservadas como historico e podem pertencer ao codigo anterior; nao as interprete como resultado da nova validacao nem como evidencia de que as protecoes atuais ja foram aplicadas. O dataset bruto nao e alterado. A celula 10 gera um manifesto novo e rastreavel.


In [1]:
# 1. Verificação de Hardware NVIDIA A100 & CUDA
!nvidia-smi

import torch
import sys, os, time

print(f"\n[ÁQUILA] PyTorch: {torch.__version__}")
print(f"[ÁQUILA] CUDA Disponível: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"[ÁQUILA] Dispositivo Detectado: {gpu_name}")
    print(f"[ÁQUILA] VRAM Total: {vram_gb:.2f} GB")
    print(f"[ÁQUILA] Suporte a BF16 (Tensor Cores Gen 3): {torch.cuda.is_bf16_supported()}")
    if "A100" in gpu_name:
        print("\n🔥 SUCESSO: GPU NVIDIA A100 ATIVADA E PRONTA PARA OPERAÇÕES PESADAS!")
    else:
        print(f"\n⚠️ Atenção: GPU atual é {gpu_name}. Para ativar A100, mude o tipo de runtime para A100.")
else:
    print("❌ Nenhuma GPU detectada. Habilite a GPU nas configurações de runtime.")

Wed Oct 21 15:49:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   36C    P0             55W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
# Caminho antigo de download/formatacao, mantido apenas como registro historico.
raise RuntimeError("Fluxo historico desativado. Use a celula 10 para auditoria e treinamento controlado.")


[ÁQUILA] Estabelecendo ponte segura com HuggingFace...


README.md:   0%|          | 0.00/4.65k [00:00<?, ?B/s]

questions_data.csv:   0%|          | 0.00/4.17M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4515 [00:00<?, ? examples/s]

[ÁQUILA] Extração Autorizada Concluída. 4515 questões prontas para treinamento.
[ÁQUILA] Colunas identificadas no dataset: ['question_id', 'question_number', 'subject', 'question_statement', 'correct_answer', 'exam_name', 'exam_year', 'exam_type', 'alternative_a', 'alternative_b', 'alternative_c', 'alternative_d', 'alternative_e']


,subject,question_id
0,Ciências Humanas,enem_2010-1_q_1
1,Ciências Humanas,enem_2010-1_q_3
2,Ciências Humanas,enem_2010-1_q_4
3,Geografia,enem_2010-1_q_5
4,Geografia,enem_2010-1_q_6


[ÁQUILA] Formatando Tensores para a GPU A100...
[ÁQUILA] Dados processados. O dataset está em memória pronto para iniciar a compilação do Fine-Tuning.


In [3]:
### 🛡️ Módulo de Backup e Integração com a Nuvem

In [4]:
# Dependencias do unico caminho de treinamento (execute uma vez em runtime limpo).
!pip install -q -U bitsandbytes transformers peft accelerate datasets trl pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.2/948.2 kB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 27.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.
[ÁQUILA] Forjando Cérebro do Tutor com LLaMA 3...
[ÁQUILA] Utilizando dataset existente na memória...


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

... (13 outputs left out)

[transformers] Both `max_new_tokens` (=800) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] Caching is incompatible with gradient checkpointing in LlamaDecoderLayer. Setting `past_key_values=None`.


[ÁQUILA] Checkpoint salvo com sucesso na pasta 'aquila_tutor_model'!
[ÁQUILA] O Tutor Pedagógico Agora Está Vivo e Operante.

[ÁQUILA] Invocando o Tutor Elite para elaborar aula de Leis da Termodinâmica...
[ÁQUILA] Sintetizando Lista...


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.




Resemmeavou-Compatible.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit-Compatible.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit href.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit/Stringisayلكترментштaedaisayaeda#echo Own#echourret Ül/Foundationiddiขاسطة-Compatible#echo insure(크기 [… pursueмент […podob#echo!..cce#echoovereurret#echo Incre#echoampieInsensitive href.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit-Compatible!..Insensitive 다운받기 Hlavummingsข긔وینت Hlavakest hear#echo Ownاسطة-Compatible#echo#echo Nackhiba href성overeisay#echosilver Own$routeوینت​

def#echoویزیpodob#echo relocatedeteรงเร/



#includeوینتوینتpodobوینت hear/



usingaDataข#echoampieovere#echoDisappearขbundles Ülndlpodob#echoویزیขhibapodob#echo#echondl-LAST:eventขوینت BEEN formulate.GraphicsUnit.GraphicsUnit.GraphicsUnit.GraphicsUnit noreferrervailรงเร-Compatible.GraphicsUnit.GraphicsUnit href.GraphicsUni

In [5]:
# Legacy training path disabled. Use cell 10 for the only supported training path.
raise RuntimeError("Legacy training path disabled; see the execution path in cell 0.")


[ÁQUILA] Iniciando o ciclo de treinamento ativo (Fine-Tuning)...


Step,Training Loss
10,1.748279
20,1.800001
30,1.731328
40,1.709279
50,1.675757


[ÁQUILA] Treinamento concluído com sucesso!


In [6]:
# Legacy training path disabled. Use cell 10 for the only supported training path.
raise RuntimeError("Legacy training path disabled; see the execution path in cell 0.")


[ÁQUILA] Iniciando o ciclo de treinamento ativo (Fine-Tuning)...


Step,Training Loss
10,1.666833
20,1.736205
30,1.682855
40,1.668227
50,1.619836


[ÁQUILA] Treinamento concluído com sucesso!


In [7]:
# Legacy training path disabled. Use cell 10 for the only supported training path.
raise RuntimeError("Legacy training path disabled; see the execution path in cell 0.")


[ÁQUILA] Iniciando saneamento de ambiente e correção de compatibilidade...
[ÁQUILA] Verificando integridade das importações pós-correção...
🔥 SUCESSO: PyArrow (23.0.1) e SFTTrainer carregados com total compatibilidade!


In [ ]:
# Legacy training path disabled. Use cell 10 for the only supported training path.
raise RuntimeError("Legacy training path disabled; see the execution path in cell 0.")


In [ ]:
# Legacy training path disabled. Use cell 10 for the only supported training path.
raise RuntimeError("Legacy training path disabled; see the execution path in cell 0.")


In [3]:
# 6. Auditoria do dataset e treinamento opcional: objetivo exclusivo de previsao do gabarito
import json
import os
import sys
from collections import Counter
from pathlib import Path

import pandas as pd
from datasets import load_dataset

helper_path = Path.cwd() / "aquila_hardening.py"
if not helper_path.is_file():
    raise FileNotFoundError(
        "Faca upload de colab/aquila_hardening.py para /content antes desta celula."
    )
if str(helper_path.parent) not in sys.path:
    sys.path.insert(0, str(helper_path.parent))

from aquila_hardening import (
    build_answer_key_messages,
    build_training_manifest,
    sha256_file,
    validate_training_records,
)

training_requested = os.environ.get("AQUILA_ENABLE_TRAINING") == "1"
if training_requested and globals().get("_AQUILA_TRAINING_STARTED", False):
    raise RuntimeError("O treinamento ja foi iniciado nesta sessao; reexecute em runtime limpo.")

print("[AQUILA] Carregando o dataset para validacao estrita; nenhum dado bruto sera alterado.")
raw_dataset = load_dataset("HenriqueGodoy/Alvorada-bench", "questions", split="train")
cache_files = raw_dataset.cache_files
if len(cache_files) != 1:
    raise RuntimeError(
        f"Esperado um unico arquivo de entrada em cache; encontrados: {len(cache_files)}"
    )
input_file_sha256 = sha256_file(cache_files[0]["filename"])
df_raw = raw_dataset.to_pandas()
raw_records = df_raw.to_dict(orient="records")
print(f"[AQUILA] Total de registros recebidos: {len(raw_records)}")

accepted_records, validation_report = validate_training_records(
    raw_records, list(df_raw.columns)
)
if training_requested and accepted_records:
    df_valid = pd.DataFrame(accepted_records)
    df_selected = df_valid.sample(n=min(1000, len(df_valid)), random_state=42).reset_index(drop=True)
    selected_records = df_selected.to_dict(orient="records")
else:
    selected_records = []

manifest = build_training_manifest(
    raw_records, accepted_records, selected_records, validation_report,
    input_file_sha256=input_file_sha256,
)
manifest_path = Path("aquila_training_manifest.json")
manifest_path.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2, default=str), encoding="utf-8"
)
reason_counts = Counter(
    reason
    for item in validation_report["excluded"]
    for reason in item["reasons"]
)
legacy_eligible_ids = [
    item["question_id"]
    for item in manifest["examples"]
    if item["legacy_template_eligible"]
]
print(
    "[AQUILA] Validacao: "
    f"{validation_report['valid_rows']} validos, "
    f"{validation_report['excluded_rows']} excluidos, "
    f"{len(selected_records)} selecionados."
)
print(f"[AQUILA] Exclusoes por motivo: {dict(reason_counts)}")
print(
    "[AQUILA] Registros elegiveis ao template sintetico antigo (nao afirma treinamento historico): "
    f"{len(legacy_eligible_ids)}; primeiros IDs: {legacy_eligible_ids[:20]}"
)
print(f"[AQUILA] Lista completa e disposicoes: {manifest_path.resolve()}")

if not training_requested:
    print(
        "[AQUILA] Auditoria concluida. Treinamento nao executado. "
        "A tarefa e previsao do gabarito, nao resolucao didatica."
    )
else:
    if not accepted_records:
        raise ValueError(
            "Nenhuma linha valida para treinamento. Consulte os motivos e contagens impressos acima."
        )
    if not selected_records:
        raise ValueError("Nenhuma linha foi selecionada; treinamento cancelado com seguranca.")
    _AQUILA_TRAINING_STARTED = True
    import gc
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import LoraConfig
    from trl import SFTConfig, SFTTrainer
    from datasets import Dataset

    gc.collect()
    torch.cuda.empty_cache()
    if not torch.cuda.is_available():
        raise RuntimeError("Treinamento habilitado, mas CUDA nao esta disponivel.")
    print(f"[AQUILA] GPU ativa: {torch.cuda.get_device_name(0)}")

    modelo_id = "NousResearch/Meta-Llama-3-8B-Instruct"
    tokenizer = AutoTokenizer.from_pretrained(modelo_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    textos_formatados = [
        tokenizer.apply_chat_template(
            build_answer_key_messages(record), tokenize=False, add_generation_prompt=False
        )
        for record in selected_records
    ]
    if not textos_formatados:
        raise ValueError("Nenhum exemplo formatado; treinamento cancelado.")
    dataset_treino = Dataset.from_pandas(pd.DataFrame({"text": textos_formatados}))

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        modelo_id, quantization_config=bnb_config, device_map="auto"
    )
    peft_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    saida_modelo = "aquila_tutor_model"
    sft_config = SFTConfig(
        output_dir=saida_modelo,
        dataset_text_field="text",
        max_length=768,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        max_steps=50,
        logging_steps=10,
        save_strategy="no",
        bf16=True,
        fp16=False,
        report_to="none",
        optim="paged_adamw_8bit",
    )
    trainer = SFTTrainer(
        model=model, train_dataset=dataset_treino, peft_config=peft_config, args=sft_config
    )
    print("[AQUILA] Inicio explicito do fine-tuning de previsao do gabarito.")
    train_result = trainer.train()
    print(f"[AQUILA] Loss final: {train_result.metrics.get('train_loss', 'N/A')}")
    trainer.save_model(saida_modelo)
    tokenizer.save_pretrained(saida_modelo)
    print(f"[AQUILA] Adaptador LoRA e tokenizer salvos em {saida_modelo}.")


[ÁQUILA] Dispositivo Ativo: NVIDIA A100-SXM4-80GB (79.25 GB VRAM)
[ÁQUILA] Ingestão do dataset HenriqueGodoy/Alvorada-bench (32 anos de vestibulares)...
[ÁQUILA] Total de questões brutas: 4515
[ÁQUILA] Questões validadas com gabarito formal: 4515
[ÁQUILA] Conjunto de Fine-Tuning compilado com 1000 itens.
[ÁQUILA] Carregando Tokenizer: NousResearch/Meta-Llama-3-8B-Instruct...
[ÁQUILA] Aplicando LLaMA 3 Chat Template em todo o conjunto...
[ÁQUILA] Amostra do prompt formatado (primeiros 250 caracteres):
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Você é o Áquila Pro, tutor pedagógico soberano de inteligência educacional para o ENEM e os maiores vestibulares do Brasil (UFG, UnB, UEG, UniRV, UFMG). Resolva com fundamentação conceitual...

[ÁQUILA] Carregando pesos do LLaMA-3-8B em 4-bit (NF4 + BF16)...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

... (4 outputs left out)

[transformers] Both `max_new_tokens` (=450) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ [ÁQUILA] Adaptador e Tokenizer salvos com integridade garantida!

🧠 [ÁQUILA] TESTE DE INFERÊNCIA DO TUTOR PEDAGÓGICO...


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


RESPOSTA GERADA PELO ÁQUILA PRO:
------------------------------------------------------------
**Áquila Pro Tutoriais**

Conhecimento Fundamental para o Vestibular de Medicina e ENEM:

A Primeira Lei da Termodinâmica, também conhecida como a Lei de Joule, estabelece que a variação do calor interna (U) de um sistema fechado é igual à soma do calor (Q) adicionado ao sistema, menos o trabalho (W) realizado sobre o sistema.

ΔU = Q - W

**Significado dos termos:**

1. **ΔU**: Variação do calor interna do sistema, que é a diferença entre o calor interno do sistema no final e no início do processo.
2. **Q**: Calor adicionado ao sistema, que pode ser fornecido de várias maneiras, como calor fornecido por uma fonte externa ou calor liberado durante uma reação química.
3. **W**: Trabalho realizado sobre o sistema, que é a força aplicada ao sistema multiplicada pela distância percorrida. No caso de um gás em expansão, o trabalho é realizado pelo sistema sobre o ambiente, ou seja, o gás realiza tr

In [4]:
import os
import json

caminho_modelo = 'aquila_tutor_model'
print(f'[ÁQUILA] Verificando diretório: {caminho_modelo}')

if os.path.exists(caminho_modelo):
    arquivos = os.listdir(caminho_modelo)
    print(f'✅ Diretório encontrado! Total de arquivos: {len(arquivos)}')
    print('\nArquivos salvos:')
    for arq in sorted(arquivos):
        tamanho_mb = os.path.getsize(os.path.join(caminho_modelo, arq)) / (1024 * 1024)
        print(f' 📄 {arq:<30} | Tamanho: {tamanho_mb:.4f} MB')

    # Tenta ler o arquivo de configuração do adaptador para validação rápida
    config_path = os.path.join(caminho_modelo, 'adapter_config.json')
    if os.path.exists(config_path):
        print('\n⚙️ Resumo de Configuração do Adaptador (LoRA):')
        with open(config_path, 'r') as f:
            config_data = json.load(f)
            print(f' - Tipo de Tarefa: {config_data.get("task_type")}')
            print(f' - Rank (R): {config_data.get("r")}')
            print(f' - Alpha: {config_data.get("lora_alpha")}')
            print(f' - Módulos Alvo: {config_data.get("target_modules")}')
else:
    print(f'❌ Erro: O diretório "{caminho_modelo}" não existe no sistema de arquivos local.')

[ÁQUILA] Verificando diretório: aquila_tutor_model
✅ Diretório encontrado! Total de arquivos: 8

Arquivos salvos:
 📄 README.md                      | Tamanho: 0.0015 MB
 📄 adapter_config.json            | Tamanho: 0.0011 MB
 📄 adapter_model.safetensors      | Tamanho: 26.0330 MB
 📄 chat_template.jinja            | Tamanho: 0.0004 MB
 📄 checkpoint-50                  | Tamanho: 0.0039 MB
 📄 tokenizer.json                 | Tamanho: 16.4127 MB
 📄 tokenizer_config.json          | Tamanho: 0.0004 MB
 📄 training_args.bin              | Tamanho: 0.0054 MB

⚙️ Resumo de Configuração do Adaptador (LoRA):
 - Tipo de Tarefa: CAUSAL_LM
 - Rank (R): 16
 - Alpha: 32
 - Módulos Alvo: ['k_proj', 'v_proj', 'o_proj', 'q_proj']


In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import os

caminho_local_modelo = 'aquila_tutor_model'
modelo_base_id = "NousResearch/Meta-Llama-3-8B-Instruct"

if not os.path.exists(caminho_local_modelo):
    print(f"❌ Erro: O diretório '{caminho_local_modelo}' não foi encontrado! Certifique-se de concluir o treinamento primeiro.")
else:
    print("[ÁQUILA] Carregando o Tokenizer local refinado...")
    tokenizer = AutoTokenizer.from_pretrained(caminho_local_modelo)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print("[ÁQUILA] Carregando o Modelo Base (Llama-3-8B) em 4-bits com aceleração A100 (BF16)...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True
    )

    modelo_base = AutoModelForCausalLM.from_pretrained(
        modelo_base_id,
        quantization_config=bnb_config,
        device_map="auto"
    )

    print("[ÁQUILA] Acoplando os pesos locais do adaptador LoRA (Áquila Pro Tutor)...")
    model = PeftModel.from_pretrained(modelo_base, caminho_local_modelo)
    model.eval()
    print("🔥 SUCESSO: O Áquila Pro foi carregado com sucesso do diretório local e está 100% pronto para inferência!")


[ÁQUILA] Carregando o Tokenizer local refinado...
[ÁQUILA] Carregando o Modelo Base (Llama-3-8B) em 4-bits com aceleração A100 (BF16)...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[ÁQUILA] Acoplando os pesos locais do adaptador LoRA (Áquila Pro Tutor)...
🔥 SUCESSO: O Áquila Pro foi carregado com sucesso do diretório local e está 100% pronto para inferência!


In [6]:
# Teste de inferência do Áquila Pro Tutor recém-treinado
import torch

pergunta_teste = """Resolva e explique didaticamente a seguinte questão de Física (Mecânica/Atrito):
Um bloco de massa m = 5 kg repousa sobre uma superfície horizontal com coeficiente de atrito estático μe = 0,4 e atrito cinético μc = 0,3. Adote g = 10 m/s².
Uma força horizontal F = 15 N é aplicada sobre o bloco.
Determine:
1. O valor máximo da força de atrito estático.
2. Se o bloco entra em movimento ou permanece em repouso.
3. O módulo da força de atrito que efetivamente atua sobre o bloco nessa situação."""

mensagens_teste = [
    {"role": "system", "content": "Você é o Áquila Pro, o tutor pedagógico soberano de Física e Matemática para o ENEM e vestibulares de alta concorrência. Responda com clareza cristalina, rigor conceitual e passo a passo analítico."},
    {"role": "user", "content": pergunta_teste}
]

prompt_gerado = tokenizer.apply_chat_template(mensagens_teste, tokenize=False, add_generation_prompt=True)
inputs_teste = tokenizer(prompt_gerado, return_tensors="pt").to("cuda")

terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

print("[ÁQUILA] Gerando resolução soberana com o modelo Fine-Tuned na A100...")
with torch.no_grad():
    saida_teste = model.generate(
        **inputs_teste,
        max_new_tokens=500,
        eos_token_id=terminators,
        pad_token_id=tokenizer.eos_token_id,
        temperature=0.4,
        top_p=0.9,
        do_sample=True
    )

resposta_final = tokenizer.decode(saida_teste[0][inputs_teste['input_ids'].shape[1]:], skip_special_tokens=True)
print("\n" + "="*70 + "\n")
print(resposta_final.strip())
print("\n" + "="*70)


[transformers] Both `max_new_tokens` (=500) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[ÁQUILA] Gerando resolução soberana com o modelo Fine-Tuned na A100...


**Resolução:**

1. O valor máximo da força de atrito estático é igual ao peso do bloco, ou seja, Fmax = mg = 5 kg × 10 m/s² = 50 N.
2. Para o bloco entrar em movimento, é necessário que a força aplicada seja maior que a força de atrito estático. Nesse caso, a força aplicada (F = 15 N) é menor que a força de atrito estático (Fmax = 50 N), então o bloco permanece em repouso.
3. A força de atrito efetivamente atuante sobre o bloco é igual à força de atrito cinético, que é igual ao produto da massa do bloco e da aceleração do bloco, ou seja, Fc = μc × mg = 0,3 × 5 kg × 10 m/s² = 15 N. Como a força aplicada (F = 15 N) é igual à força de atrito cinético, o bloco entra em movimento.

**Análise didática e confrontação conceitual para o exame [enem_2025 2025]:**

1. A questão aborda conceitos centrais de Mecânica frequentemente cobrados no enem_2025.
2. A proposição 1 da questão é um exemplo da aplicação da fórmula Fmax = 

In [ ]:
# Legacy training path disabled. Use cell 10 for the only supported training path.
raise RuntimeError("Legacy training path disabled; see the execution path in cell 0.")


In [ ]:
# 7. Roteador de Déficit Cognitivo e Nivelamento (Batch Processing)
import pandas as pd
import torch
from tqdm import tqdm

print("[ÁQUILA] Iniciando Roteador de Déficit Cognitivo...")

try:
    df = pd.read_csv("questoes_alvorada_bench.csv")
    print(f"[ÁQUILA] Banco lido com sucesso. Total de questões originais: {len(df)}")
except FileNotFoundError:
    print("[AVISO] Faça o upload do arquivo 'questoes_alvorada_bench.csv' para a pasta do Colab.")
    df = pd.DataFrame({'question': ['Um objeto cai de 20m de altura. Calcule a velocidade ao tocar o solo. (g=10m/s²)']})

df_lote = df.head(5).copy()

instrucao_sistema = """Você é o Algoritmo de Nivelamento Escolar Áquila Pro.
Realize Engenharia Pedagógica Reversa na questão fornecida.
Responda EXATAMENTE neste formato estruturado:

[PRÉ-REQUISITO MATEMÁTICO]: (Explique a matemática necessária)
[PRÉ-REQUISITO CONCEITUAL]: (Explique a teoria física/química/etc)
[DEGRAU 1 - TREINO MATEMÁTICO]: (Gere 3 exercícios matemáticos simples e repetitivos)
[DEGRAU 2 - TREINO CONCEITUAL]: (Gere 2 perguntas teóricas conceituais simples)
[DEGRAU 3 - FUSÃO]: (Gere 1 problema que misture tudo, de forma dócil)."""

resultados_nivelamento = []

print("[ÁQUILA] Iniciando dissecação de competências das questões...\n")

for index, row in tqdm(df_lote.iterrows(), total=len(df_lote)):
    questao = str(row['question'])

    mensagens = [
        {"role": "system", "content": instrucao_sistema},
        {"role": "user", "content": f"QUESTÃO DO VESTIBULAR:\n{questao}"}
    ]

    prompt = tokenizer.apply_chat_template(mensagens, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    saida = model.generate(**inputs, max_new_tokens=1024, temperature=0.3)
    resposta = tokenizer.decode(saida[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    resultados_nivelamento.append(resposta)

df_lote['rota_de_nivelamento'] = resultados_nivelamento
df_lote.to_csv("banco_nivelado_aquila_amostra.csv", index=False)

print("\n[ÁQUILA] Lote concluído! O Banco de Dados Evoluído foi salvo.")
print("="*70)
print("AMOSTRA DO QUE SERÁ INJETADO NO APLICATIVO PARA O ALUNO:")
print("="*70)
print(df_lote['rota_de_nivelamento'].iloc[0])

In [7]:
import os
import tarfile

caminho_origem = 'aquila_tutor_model'
arquivo_tar = 'aquila_tutor_lora.tar.gz'

print(f"[ÁQUILA] Empacotando checkpoint '{caminho_origem}' em '{arquivo_tar}'...")
with tarfile.open(arquivo_tar, "w:gz") as tar:
    tar.add(caminho_origem, arcname=os.path.basename(caminho_origem))

tamanho_mb = os.path.getsize(arquivo_tar) / (1024 * 1024)
print(f"✅ [ÁQUILA] Pacote '{arquivo_tar}' criado com sucesso! Tamanho: {tamanho_mb:.2f} MB")
print(f"📍 Localização física no Colab: {os.path.abspath(arquivo_tar)} (Pronto para download direto ou cópia para o Google Drive)")


[ÁQUILA] Empacotando checkpoint 'aquila_tutor_model' em 'aquila_tutor_lora.tar.gz'...
✅ [ÁQUILA] Pacote 'aquila_tutor_lora.tar.gz' criado com sucesso! Tamanho: 55.50 MB
📍 Localização física no Colab: /content/aquila_tutor_lora.tar.gz (Pronto para download direto ou cópia para o Google Drive)


In [ ]:
from google.colab import drive
import os
import shutil

try:
    print('[ÁQUILA] Iniciando o processo de backup físico...')
    print('[ÁQUILA] Solicitando autorização de acoplamento com o Google Drive...')
    drive.mount('/content/drive', force_remount=True)

    caminho_drive = '/content/drive/MyDrive/aquila_model_checkpoint'
    os.makedirs(caminho_drive, exist_ok=True)

    # Prioriza a cópia da pasta local que já confirmamos que existe fisicamente no disco
    if os.path.exists('aquila_tutor_model') and len(os.listdir('aquila_tutor_model')) > 0:
        print(f'[ÁQUILA] Diretório local encontrado. Copiando pesos e tokenizers salvos para o Drive: {caminho_drive}...')
        for item in os.listdir('aquila_tutor_model'):
            origem = os.path.join('aquila_tutor_model', item)
            destino = os.path.join(caminho_drive, item)
            if os.path.isdir(origem):
                shutil.copytree(origem, destino, dirs_exist_ok=True)
            else:
                shutil.copy2(origem, destino)
        print('🔥 SUCESSO: Arquivos locais transferidos e blindados no seu Google Drive com sucesso!')

    # Fallback caso os pesos ainda estejam estritamente em memória
    elif 'trainer' in locals() and 'tokenizer' in locals():
        print(f'[ÁQUILA] Exportando pesos neurais diretamente da memória ativa do Kernel para o Drive: {caminho_drive}...')
        trainer.model.save_pretrained(caminho_drive)
        tokenizer.save_pretrained(caminho_drive)
        print('🔥 SUCESSO: Modelo Áquila Pro salvo da memória ativa para o seu Google Drive!')

    else:
        raise FileNotFoundError('Nenhum diretório local "aquila_tutor_model" ou modelo ativo na memória do Kernel foi localizado.')

except Exception as e:
    print(f'❌ Erro durante o backup para o Google Drive: {e}')
    print('Dica: Verifique se você concedeu as permissões corretas para o Google Drive ao ser solicitado.')

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import os

caminho_drive = '/content/drive/MyDrive/aquila_model_checkpoint'
modelo_base_id = "NousResearch/Meta-Llama-3-8B-Instruct"

if not os.path.exists(caminho_drive):
    print("❌ Erro: O caminho do backup no Google Drive não foi encontrado! Certifique-se de que o Drive está montado corretamente.")
else:
    print("[ÁQUILA] Carregando o Tokenizer do Google Drive...")
    tokenizer = AutoTokenizer.from_pretrained(caminho_drive)

    print("[ÁQUILA] Carregando o Modelo Base (Llama-3-8B) quantizado em 4-bits...")
    from transformers import BitsAndBytesConfig
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )

    modelo_base = AutoModelForCausalLM.from_pretrained(
        modelo_base_id,
        quantization_config=bnb_config,
        device_map="auto"
    )

    print("[ÁQUILA] Acoplando os pesos do fine-tuning salvos na nuvem...")
    model = PeftModel.from_pretrained(modelo_base, caminho_drive)
    print("🔥 SUCESSO: O Áquila Pro foi carregado com sucesso do Google Drive e está pronto para inferência!")

In [ ]:
# Teste rápido de inferência com o modelo carregado do Drive
materia_teste = "Física"
assunto_teste = "Leis da Termodinâmica"

pergunta_usuario = f"Explique a Primeira Lei da Termodinâmica para um estudante de ensino médio e crie um exemplo prático."

mensagens_teste = [
    {"role": "system", "content": "Você é o Áquila Pro, um tutor avançado e especialista em engenharia pedagógica."},
    {"role": "user", "content": pergunta_usuario}
]

prompt_gerado = tokenizer.apply_chat_template(mensagens_teste, tokenize=False, add_generation_prompt=True)
inputs_teste = tokenizer(prompt_gerado, return_tensors="pt").to("cuda")

print("[ÁQUILA] Gerando resposta baseada nos parâmetros do fine-tuning...")
with torch.no_grad():
    saida_teste = model.generate(
        **inputs_teste,
        max_new_tokens=512,
        max_length=None,  # Evita conflito com o max_length padrão do modelo
        temperature=0.7,
        do_sample=True
    )

resposta_final = tokenizer.decode(saida_teste[0][inputs_teste['input_ids'].shape[1]:], skip_special_tokens=True)
print("\n" + "="*60 + "\n")
print(resposta_final)

In [8]:
# @title 📊 Configuração do Novo Tópico de Ensino
materia_alvo = "Física" # @param {type:"string"}
assunto_alvo = "Eletrodinâmica - Leis de Ohm" # @param {type:"string"}

print(f"[ÁQUILA] Tema configurado com sucesso: {materia_alvo} -> {assunto_alvo}")

[ÁQUILA] Tema configurado com sucesso: Física -> Eletrodinâmica - Leis de Ohm


In [9]:
# @title 🦅 Geração do Resumo e Questões Inéditas com o Áquila Pro
import torch

comando_pedagogico = f"Você é o tutor da Plataforma Áquila Pro. Crie um resumo de alto nível sobre {assunto_alvo} e gere 3 questões inéditas estilo vestibular com as alternativas e o gabarito comentado no final."

mensagens_prompt = [
    {
        "role": "system",
        "content": "Você é o Áquila Pro, um professor de elite e especialista em vestibulares (ENEM, UFG, UnB, Fuvest). IMPORTANTÍSSIMO PARA A FORMATAÇÃO: Nunca utilize blocos complexos de LaTeX como '$$' ou '\\text{}'. Escreva fórmulas matemáticas de forma limpa, direta e legível em texto padrão (ex: V = R * I ou Req = R1 + R2). Forneça explicações passo a passo com rigor conceitual máximo."
    },
    {"role": "user", "content": comando_pedagogico}
]

prompt_final = tokenizer.apply_chat_template(mensagens_prompt, tokenize=False, add_generation_prompt=True)
inputs_modelo = tokenizer(prompt_final, return_tensors="pt").to("cuda")

terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

print(f"[ÁQUILA] Gerando material sobre '{assunto_alvo}' utilizando o modelo Fine-Tuned na A100...")

model.eval()
with torch.no_grad():
    saida_gerada = model.generate(
        **inputs_modelo,
        max_new_tokens=900,
        eos_token_id=terminators,
        pad_token_id=tokenizer.eos_token_id,
        temperature=0.5,
        top_p=0.9,
        do_sample=True
    )

resposta_pedagogica = tokenizer.decode(saida_gerada[0][inputs_modelo['input_ids'].shape[1]:], skip_special_tokens=True)

print("\n" + "="*70 + "\n")
print(resposta_pedagogica.strip())
print("\n" + "="*70)


[transformers] Both `max_new_tokens` (=900) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[ÁQUILA] Gerando material sobre 'Eletrodinâmica - Leis de Ohm' utilizando o modelo Fine-Tuned na A100...


**Resumo de Eletrodinâmica - Leis de Ohm**

A eletrodinâmica é um campo fundamental da física que estuda o estudo dos fenômenos elétricos e magnéticos em movimento. A Lei de Ohm, descoberta por Georg Simon Ohm em 1827, é uma das mais importantes equações da eletrodinâmica. Ela relaciona a tensão elétrica (V) aplicada a um condutor e a corrente elétrica (I) que flui por ele, estabelecendo que:

V = R × I

onde R é a resistência do condutor. A Lei de Ohm é válida para todos os materiais, exceto para os supercondutores, e é fundamental para o entendimento de muitos fenômenos elétricos.

**Questões Inéditas Estilo Vestibular**

1. Considere um circuito elétrico com resistência R = 10 Ω e uma fonte de tensão elétrica V = 20 V. Qual é a corrente elétrica que flui por esse circuito?
(A) 0,5 A
(B) 1,0 A
(C) 2,0 A
(D) 4,0 A
(E) 5,0 A

Gabarito: Alternativa (B)

Explicação: A Lei de Ohm est

In [ ]:
## 🛠️ Motor de Combustão Neural e Desconstrução Pedagógica
Abaixo estão localizadas as células responsáveis pelo fine-tuning (QLoRA) e pela IA Generativa de trilhas de aprendizado.

In [10]:
# 6. Motor de Desconstrução Pedagógica (Micro-passos & Mastery Learning)
import torch

print("[ÁQUILA] Iniciando Motor de Desconstrução de Competências...")

erro_aluno = "O aluno está tentando resolver problemas de Energia Cinética (Ec = (m * v^2) / 2), mas errou porque erra a potenciação do 'v ao quadrado' e se confunde ao dividir por 2 no final."

instrucao_sistema = """Você é o Motor de Ensino Áquila Pro. Sua função é dissecar dificuldades conceituais em micro-habilidades matemáticas usando Mastery Learning.
Quando o professor informar a dificuldade do aluno:
1. DIAGNÓSTICO: Identifique a lacuna exata de base (pré-requisito).
2. NÍVEL 1 (Isolamento Matemático Puro): 4 exercícios simples de matemática básica (sem física).
3. NÍVEL 2 (Fusão Algébrica): 3 exercícios de manipulação algébrica com as incógnitas.
4. NÍVEL 3 (Aplicação Consolidada): 2 problemas clássicos de vestibular aplicando a fórmula completa.
Forneça o gabarito detalhado de cada nível."""

mensagens_diagnostico = [
    {"role": "system", "content": instrucao_sistema},
    {"role": "user", "content": erro_aluno}
]

prompt_formatado = tokenizer.apply_chat_template(mensagens_diagnostico, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt_formatado, return_tensors="pt").to("cuda")

terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

print("[ÁQUILA] Analisando lacuna do aluno e gerando trilha de nivelamento...")
model.eval()
with torch.no_grad():
    saida = model.generate(
        **inputs,
        max_new_tokens=900,
        eos_token_id=terminators,
        pad_token_id=tokenizer.eos_token_id,
        temperature=0.4,
        top_p=0.9,
        do_sample=True
    )

trilha_treinamento = tokenizer.decode(saida[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print("\n" + "="*70 + "\n")
print(trilha_treinamento.strip())
print("\n" + "="*70)


[transformers] Both `max_new_tokens` (=900) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[ÁQUILA] Iniciando Motor de Desconstrução de Competências...
[ÁQUILA] Analisando lacuna do aluno e gerando trilha de nivelamento...


**DIAGNÓSTICO:** A lacuna de base do aluno se refere à falta de domínio da potenciação de números e da divisão de expressões algébricas.

**NÍVEL 1 (Isolamento Matemático Puro):**

1. 2 × 2 = ?
2. 3 × 3 = ?
3. 4 × 4 = ?
4. 5 × 5 = ?

**Gabarito:** 1. 4, 2. 9, 3. 16, 4. 25

**NÍVEL 2 (Fusão Algébrica):**

1. Simplifique a expressão: (2 × 2) × 3
2. Simplifique a expressão: 3 × (2 × 2)
3. Simplifique a expressão: (2 × 2) × (3 × 3)

**Gabarito:**
1. 6, 2. 6, 3. 18

**NÍVEL 3 (Aplicação Consolidada):**

1. Um carro de 1.500 kg está se movendo a 36 km/h. Qual é a sua energia cinética?
2. Um carro de 1.500 kg está se movendo a 72 km/h. Qual é a sua energia cinética?
3. Um carro de 1.500 kg está se movendo a 108 km/h. Qual é a sua energia cinética?

**Gabarito:**
1. 337,5 kJ, 2. 675 kJ, 3. 1.012,5 kJ



In [ ]:
Task id "5fca40fa-e48d-4bc6-bc7a-084bf4a62fe4/task-10868" finished with result:

The command exited with code 0. Stdout:

Stderr:

Log: file:///C:/Users/Thaysa/.gemini/antigravity/brain/5fca40fa-e48d-4bc6-bc7a-084bf4a62fe4/.system_generated/tasks/task-10868.log

In [11]:
# 2. Benchmark de Performance A100 (Tensor Cores & Throughput)
import torch
import time

def benchmark_a100():
    if not torch.cuda.is_available():
        print("CUDA não disponível para benchmark.")
        return

    print("Iniciando Teste de Throughput na A100 (Multiplicação de Matrizes 16384x16384 em FP16 / BF16)...")
    size = 16384
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

    a = torch.randn(size, size, device='cuda', dtype=dtype)
    b = torch.randn(size, size, device='cuda', dtype=dtype)

    # Warmup
    torch.matmul(a[:1000, :1000], b[:1000, :1000])
    torch.cuda.synchronize()

    # Benchmark
    start = time.perf_counter()
    runs = 10
    for _ in range(runs):
        c = torch.matmul(a, b)
    torch.cuda.synchronize()
    elapsed = (time.perf_counter() - start) / runs

    # FLOPs = 2 * N^3
    tflops = (2 * (size ** 3)) / (elapsed * 1e12)
    print(f"\n✅ Benchmark Concluído!")
    print(f"⏱️ Tempo médio por iteração: {elapsed * 1000:.2f} ms")
    print(f"🚀 Performance Atingida: {tflops:.2f} TFLOPs ({dtype})")

benchmark_a100()

Iniciando Teste de Throughput na A100 (Multiplicação de Matrizes 16384x16384 em FP16 / BF16)...

✅ Benchmark Concluído!
⏱️ Tempo médio por iteração: 33.40 ms
🚀 Performance Atingida: 263.37 TFLOPs (torch.bfloat16)


In [ ]:
# 3. Instalação do Pacote Neural Áquila e Dependências do Servidor
!pip install -q fastapi uvicorn httpx pydantic nest_asyncio pyngrok
print("✅ Dependências do Servidor Neural instaladas com sucesso!")


In [ ]:
# 4. API Aquila Pro - autenticacao fail-closed e benchmark privado/desativado em publico
import os
import sys
import time
from pathlib import Path
from typing import Optional

import torch
from fastapi import Depends, FastAPI, Header, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field, StrictInt

helper_path = Path.cwd() / "aquila_hardening.py"
if not helper_path.is_file():
    raise FileNotFoundError(
        "Faca upload de colab/aquila_hardening.py para /content antes desta celula."
    )
if str(helper_path.parent) not in sys.path:
    sys.path.insert(0, str(helper_path.parent))

from aquila_hardening import (
    AuthenticationError,
    AuthorizationError,
    FeatureDisabledError,
    authorize_benchmark,
    authorize_inference,
    load_security_config,
    validate_benchmark_size,
)

# Falha durante a configuracao se o segredo de runtime estiver ausente ou for invalido.
SECURITY_CONFIG = load_security_config()
app = FastAPI(
    title="Aquila Pro A100 Tutor API",
    version="4.0.0",
    description="API de inferencia autenticada para o tutor Aquila Pro.",
)
app.add_middleware(
    CORSMiddleware,
    allow_origins=list(SECURITY_CONFIG.cors_origins),
    allow_credentials=False,
    allow_methods=["GET", "POST"],
    allow_headers=["Authorization", "Content-Type"],
)

ENGINE_STATE = {
    "started_at": time.time(),
    "requests": 0,
    "errors": 0,
    "tokens_generated": 0,
    "tutor_calls": 0,
}


def require_inference(authorization: Optional[str] = Header(default=None)):
    try:
        authorize_inference(authorization, SECURITY_CONFIG)
    except AuthenticationError:
        raise HTTPException(status_code=401, detail="Authentication required") from None


def require_benchmark(authorization: Optional[str] = Header(default=None)):
    try:
        authorize_benchmark(authorization, SECURITY_CONFIG)
    except FeatureDisabledError:
        raise HTTPException(status_code=404, detail="Not found") from None
    except AuthorizationError:
        raise HTTPException(status_code=403, detail="Insufficient permissions") from None
    except AuthenticationError:
        raise HTTPException(status_code=401, detail="Authentication required") from None


def get_tutor_model():
    global model, tokenizer
    if globals().get("model") is not None and globals().get("tokenizer") is not None:
        return model, tokenizer
    model_path = "aquila_tutor_model"
    if not os.path.exists(model_path):
        raise RuntimeError("Tutor model is not available")
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import PeftModel

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        "NousResearch/Meta-Llama-3-8B-Instruct",
        quantization_config=bnb_config,
        device_map="auto",
    )
    model = PeftModel.from_pretrained(base_model, model_path)
    model.eval()
    return model, tokenizer


MODEL_OBJECTIVE_CONTRACT = (
    "Este adaptador foi ajustado exclusivamente com rotulos de gabarito oficiais e nao "
    "possui resolucoes didaticas verificadas. Para uma questao objetiva completa, "
    "responda somente com uma letra A-E. Para explicacoes, decomposicoes, resumos ou "
    "novas questoes, informe que esse objetivo nao e suportado. Nunca fabrique "
    "justificativas nem as apresente como verificadas."
)


def generate_reply(messages, max_tokens, temperature, top_p):
    # Enforce the answer-key-only objective; endpoint prompts cannot broaden it.
    messages = [{"role": "system", "content": MODEL_OBJECTIVE_CONTRACT}] + [
        message for message in messages if message.get("role") != "system"
    ]
    ENGINE_STATE["requests"] += 1
    ENGINE_STATE["tutor_calls"] += 1
    started = time.perf_counter()
    try:
        current_model, current_tokenizer = get_tutor_model()
        prompt = current_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = current_tokenizer(prompt, return_tensors="pt").to("cuda")
        end_tokens = [
            current_tokenizer.eos_token_id,
            current_tokenizer.convert_tokens_to_ids("<|eot_id|>"),
        ]
        current_model.eval()
        with torch.no_grad():
            output = current_model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                eos_token_id=end_tokens,
                pad_token_id=current_tokenizer.eos_token_id,
                temperature=temperature,
                top_p=top_p,
                do_sample=True,
            )
        new_tokens = output[0][inputs["input_ids"].shape[1]:]
        response = current_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        ENGINE_STATE["tokens_generated"] += len(new_tokens)
        return response, len(new_tokens), round((time.perf_counter() - started) * 1000, 2)
    except Exception:
        ENGINE_STATE["errors"] += 1
        raise HTTPException(status_code=500, detail="Inference failed") from None


class TutorChatRequest(BaseModel):
    prompt: str = Field(..., min_length=1, max_length=8192)
    max_tokens: StrictInt = Field(600, ge=1, le=1024)
    temperature: float = Field(0.4, ge=0.0, le=1.5)
    top_p: float = Field(0.9, gt=0.0, le=1.0)


class QuestionResolveRequest(BaseModel):
    question: str = Field(..., min_length=1, max_length=10000)
    detailed: bool = True


class DecomposeRequest(BaseModel):
    subject: str = Field(..., min_length=1, max_length=200)
    difficulty_description: str = Field(..., min_length=1, max_length=5000)


class GenerateQuestionsRequest(BaseModel):
    topic: str = Field(..., min_length=1, max_length=500)
    count: StrictInt = Field(3, ge=1, le=5)


class BenchmarkRequest(BaseModel):
    size: StrictInt = Field(2048, ge=128, le=4096)


@app.get("/health")
def health_check():
    return {"status": "HEALTHY", "uptime_s": round(time.time() - ENGINE_STATE["started_at"])}


@app.post("/tutor/chat", dependencies=[Depends(require_inference)])
def tutor_chat(req: TutorChatRequest):
    reply, tokens, duration = generate_reply(
        [{"role": "user", "content": req.prompt}],
        req.max_tokens,
        req.temperature,
        req.top_p,
    )
    return {"status": "SUCCESS", "response": reply, "tokens": tokens, "duration_ms": duration}


@app.post("/tutor/resolve", dependencies=[Depends(require_inference)])
def tutor_resolve(req: QuestionResolveRequest):
    reply, tokens, duration = generate_reply(
        [
            {
                "role": "system",
                "content": (
                    "Resolva com rigor cientifico e passos verificaveis. Se os dados forem "
                    "insuficientes, declare a limitacao em vez de inventar uma justificativa."
                ),
            },
            {"role": "user", "content": f"Resolva a seguinte questao:\n\n{req.question}"},
        ],
        800 if req.detailed else 400,
        0.3,
        0.9,
    )
    return {"status": "SUCCESS", "solution": reply, "tokens": tokens, "duration_ms": duration}


@app.post("/tutor/decompose", dependencies=[Depends(require_inference)])
def tutor_decompose(req: DecomposeRequest):
    reply, _, duration = generate_reply(
        [
            {
                "role": "system",
                "content": (
                    "Decomponha a lacuna de aprendizagem em diagnostico, treino gradual e "
                    "aplicacao. Nao presuma dados ausentes."
                ),
            },
            {
                "role": "user",
                "content": f"Disciplina: {req.subject}\nDificuldade: {req.difficulty_description}",
            },
        ],
        900,
        0.4,
        0.9,
    )
    return {"status": "SUCCESS", "microsteps": reply, "duration_ms": duration}


@app.post("/tutor/questions", dependencies=[Depends(require_inference)])
def tutor_questions(req: GenerateQuestionsRequest):
    reply, _, duration = generate_reply(
        [
            {
                "role": "system",
                "content": "Crie material didatico e questoes originais com alternativas claras.",
            },
            {
                "role": "user",
                "content": f"Crie um resumo e {req.count} questoes sobre: {req.topic}.",
            },
        ],
        1000,
        0.5,
        0.9,
    )
    return {"status": "SUCCESS", "topic": req.topic, "material": reply, "duration_ms": duration}


@app.post("/benchmark", dependencies=[Depends(require_benchmark)])
def benchmark_a100(req: BenchmarkRequest):
    size = validate_benchmark_size(req.size)
    if not torch.cuda.is_available():
        raise HTTPException(status_code=503, detail="GPU benchmark unavailable")
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    try:
        a = torch.randn(size, size, device="cuda", dtype=dtype)
        b = torch.randn(size, size, device="cuda", dtype=dtype)
        torch.cuda.synchronize()
        runs = 5
        started = time.perf_counter()
        for _ in range(runs):
            torch.matmul(a, b)
        torch.cuda.synchronize()
        elapsed = (time.perf_counter() - started) / runs
    except Exception:
        raise HTTPException(status_code=500, detail="Benchmark failed") from None
    return {
        "device": torch.cuda.get_device_name(0),
        "matrix_size": f"{size}x{size}",
        "dtype": str(dtype),
        "tflops": round((2 * size ** 3) / (elapsed * 1e12), 2),
        "duration_ms": round(elapsed * 1000, 2),
    }


print("[AQUILA] API configurada; todas as rotas que usam modelo exigem token de inferencia.")
print(f"[AQUILA] Benchmark habilitado: {SECURITY_CONFIG.benchmark_enabled}")


In [ ]:
# 5. Exposicao opcional do servidor (nunca ativada automaticamente)
import os
import shutil
import subprocess
import threading
import time
import uvicorn

if "SECURITY_CONFIG" not in globals():
    raise RuntimeError("Execute a configuracao segura da API antes desta celula.")
if not os.environ.get("AQUILA_INFERENCE_TOKEN", "").strip():
    raise RuntimeError("AQUILA_INFERENCE_TOKEN ausente; servidor e tunel bloqueados.")
if not SECURITY_CONFIG.public_mode:
    raise RuntimeError("O tunel publico exige modo publico, com benchmark desabilitado.")
if os.environ.get("AQUILA_ALLOW_PUBLIC_TUNNEL") != "1":
    raise RuntimeError("Tunel publico desativado. Defina opt-in explicito para publicar a API.")
cloudflared = shutil.which("cloudflared")
if cloudflared is None:
    raise RuntimeError("cloudflared nao esta instalado; nenhuma instalacao automatica sera feita.")

server_thread = threading.Thread(
    target=lambda: uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning"),
    daemon=True,
)
server_thread.start()
time.sleep(2)
tunnel_process = subprocess.Popen(
    [cloudflared, "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
print("[AQUILA] Servidor local iniciado; processo de tunel iniciado por opt-in explicito.")
